## steam player

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Save models
import joblib


In [ ]:
df = pd.read_csv("../data/games.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## Data Preparation & Feature Selection

In [ ]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

df["release_year"] = df["release_date"].dt.year

In [ ]:
df["release_month"] = df["release_date"].dt.month

In [ ]:
df[
    [
        "release_date",
        "release_year",
        "release_month"
    ]
].head()

In [ ]:
df[
    [
        "release_year",
        "release_month"
    ]
].isnull().sum()

In [ ]:
cluster_features = [
    "price",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "achievements",
    "dlc_count",
    "peak_ccu",
    "metacritic_score",
    "release_year"
]

In [ ]:
cluster_df = df[cluster_features].copy()

In [ ]:
cluster_df.head()

In [ ]:
cluster_df.info()

In [ ]:
cluster_df.describe()

In [ ]:
cluster_df.isnull().sum()

In [ ]:
cluster_df = cluster_df.dropna()

In [ ]:
cluster_df.isnull().sum()

## Feature Scaling

In [ ]:
cluster_df.describe().T[
    ["min", "max", "mean", "std"]
]

In [ ]:
plt.figure(figsize=(12,6))

cluster_df.boxplot(rot=90)

plt.title("Feature Distribution Before Scaling")

plt.tight_layout()

plt.savefig(
    "../screenshots/features_before_scaling.png",
    dpi=300
)

plt.show()

In [ ]:
cluster_df.skew().sort_values(ascending=False)

In [ ]:
log_features = [
    "price",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "achievements",
    "dlc_count",
    "peak_ccu"
]

In [ ]:
cluster_df[log_features] = np.log1p(cluster_df[log_features])

In [ ]:
cluster_df[log_features].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["positive"], bins=50)

plt.title("Positive Reviews Before Log Transform")

plt.tight_layout()

plt.savefig(
    "../screenshots/positive_before_log.png",
    dpi=300
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(cluster_df["positive"], bins=50)

plt.title("Positive Reviews After Log Transform")

plt.tight_layout()

plt.savefig(
    "../screenshots/positive_after_log.png",
    dpi=300
)

plt.show()

In [ ]:
scaler = StandardScaler()

In [ ]:
scaled_features = scaler.fit_transform(cluster_df)

In [ ]:
scaled_df = pd.DataFrame(
    scaled_features,
    columns=cluster_df.columns
)

In [ ]:
scaled_df.head()

In [ ]:
scaled_df.describe().T[
    ["mean", "std", "min", "max"]
]

In [ ]:
plt.figure(figsize=(12,6))

scaled_df.boxplot(rot=90)

plt.title("Feature Distribution After Standard Scaling")

plt.tight_layout()

plt.savefig(
    "../screenshots/features_after_scaling.png",
    dpi=300
)

plt.show()

## Optimal Number of Clusters

In [ ]:
inertia = []

In [ ]:
k_values = range(2, 11)

In [ ]:
for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(scaled_df)

    inertia.append(model.inertia_)

In [ ]:
pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertia
})

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    k_values,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.grid(True)

plt.tight_layout()

plt.savefig(
    "../screenshots/elbow_method.png",
    dpi=300
)

plt.show()

In [ ]:
silhouette_scores = []

In [ ]:
for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(scaled_df)

    score = silhouette_score(
        scaled_df,
        labels
    )

    silhouette_scores.append(score)

In [ ]:
pd.DataFrame({
    "K": list(k_values),
    "Silhouette Score": silhouette_scores
})

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis")

plt.grid(True)

plt.tight_layout()

plt.savefig(
    "../screenshots/silhouette_scores.png",
    dpi=300
)

plt.show()

In [ ]:
#compare
evaluation_df = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertia,
    "Silhouette Score": silhouette_scores
})

evaluation_df

## PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(scaled_df)

print(X_pca.shape)

In [ ]:
explained_variance = pca.explained_variance_ratio_

print(explained_variance)
print("Total Explained Variance:", explained_variance.sum())

## K-Means Model

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(scaled_df)

In [ ]:
df_clustered = df.copy()

df_clustered["Cluster"] = clusters

In [ ]:
df_clustered.head()

In [ ]:
cluster_counts = (
    df_clustered["Cluster"]
    .value_counts()
    .sort_index()
)

cluster_counts

In [ ]:
cluster_percentages = (
    df_clustered["Cluster"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

cluster_percentages

In [ ]:
cluster_summary = pd.DataFrame({
    "Games": cluster_counts,
    "Percentage": cluster_percentages.round(2)
})

cluster_summary

In [ ]:
plt.figure(figsize=(8,5))

cluster_counts.plot(kind="bar")

plt.title("Number of Games per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Games")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig(
    "../screenshots/cluster_distribution.png",
    dpi=300
)

plt.show()

## Cluster Profiling (Business Intelligence)

In [ ]:
profile_features = [
    "price",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "achievements",
    "dlc_count",
    "peak_ccu",
    "metacritic_score"
]

In [ ]:
cluster_profile = (
    df_clustered
    .groupby("Cluster")[profile_features]
    .mean()
    .round(2)
)

cluster_profile

In [ ]:
cluster_profile["Games"] = cluster_counts

cluster_profile

In [ ]:
cluster_profile = cluster_profile[
    [
        "Games",
        "price",
        "positive",
        "negative",
        "recommendations",
        "average_playtime_forever",
        "achievements",
        "dlc_count",
        "peak_ccu",
        "metacritic_score"
    ]
]

cluster_profile

In [ ]:
cluster_profile.to_csv(
    "../results/cluster_profile.csv"
)

In [ ]:
plt.figure(figsize=(12,6))

plt.imshow(
    cluster_profile.drop(columns="Games"),
    aspect="auto"
)

plt.colorbar(label="Average Value")

plt.xticks(
    range(len(cluster_profile.columns)-1),
    cluster_profile.columns[1:],
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(cluster_profile.index)),
    cluster_profile.index
)

plt.title("Cluster Feature Heatmap")

plt.tight_layout()

plt.savefig(
    "../screenshots/cluster_heatmap.png",
    dpi=300
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

cluster_profile["price"].plot(kind="bar")

plt.title("Average Price per Cluster")
plt.ylabel("Price ($)")
plt.xlabel("Cluster")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig(
    "../screenshots/cluster_price.png",
    dpi=300
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

cluster_profile["positive"].plot(kind="bar")

plt.title("Average Positive Reviews")
plt.ylabel("Reviews")
plt.xlabel("Cluster")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig(
    "../screenshots/cluster_positive_reviews.png",
    dpi=300
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

cluster_profile["average_playtime_forever"].plot(kind="bar")

plt.title("Average Playtime")
plt.ylabel("Minutes")
plt.xlabel("Cluster")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig(
    "../screenshots/cluster_playtime.png",
    dpi=300
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

cluster_profile["recommendations"].plot(kind="bar")

plt.title("Average Recommendations")
plt.ylabel("Recommendations")
plt.xlabel("Cluster")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig(
    "../screenshots/cluster_recommendations.png",
    dpi=300
)

plt.show()

In [ ]:
cluster_names = {
    0: "Small Indie Games",
    1: "Growing Community Games",
    2: "Premium AAA Hits",
    3: "Long-Term Engagement Games",
    4: "Well-Received Premium Games"
}

df_clustered["Cluster_Name"] = (
    df_clustered["Cluster"]
    .map(cluster_names)
)

df_clustered[["name", "Cluster", "Cluster_Name"]].head()

In [ ]:
df_clustered.to_csv(
    "../results/steam_game_clusters.csv",
    index=False
)

In [ ]:
pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = clusters

pca_df.head()

In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    c=pca_df["Cluster"],
    s=5,
    alpha=0.5
)

plt.title("Steam Game Clusters (PCA Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.tight_layout()

plt.savefig(
    "../screenshots/pca_clusters.png",
    dpi=300
)

plt.show()

In [ ]:
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

joblib.dump(
    pca,
    "../models/pca.pkl"
)

joblib.dump(
    kmeans,
    "../models/kmeans.pkl"
)

print("Models saved successfully!")

In [ ]:
explained_variance = pca.explained_variance_ratio_

print("Explained Variance Ratio:")
print(explained_variance)

print(f"\nTotal Explained Variance: {explained_variance.sum():.2%}")

In [ ]:
plt.figure(figsize=(6,5))

plt.bar(
    ["PC1", "PC2"],
    explained_variance
)

plt.title("Explained Variance by Principal Component")
plt.ylabel("Explained Variance Ratio")

plt.tight_layout()

plt.savefig("../screenshots/pca_variance.png", dpi=300)

plt.show()